# Gradient Boosting Trio: XGBoost vs LightGBM vs CatBoost

**Learning objectives:** Train three gradient-boosted tree classifiers on the same synthetic data, compare accuracy and wall time, and skim feature importance side by side.

## Introduction

**Gradient boosting** fits an ensemble of shallow decision trees sequentially: each new tree predicts the residual (or gradient of the loss) of the current model, and predictions are summed with a learning rate. That yields strong accuracy on tabular data with moderate tuning.

This notebook compares three popular implementations:

| Library | Typical strengths |
|---------|-------------------|
| **XGBoost** | Mature ecosystem, regularization, widespread deployment |
| **LightGBM** | Leaf-wise growth, often fast on large numeric tables |
| **CatBoost** | Strong defaults with categorical features, ordered boosting |

All steps use **synthetic** data from `sklearn.datasets.make_classification` so you can run without external files.

In [ ]:
import time

import numpy as np

try:
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score
except ImportError:
    print("Install: pip install scikit-learn numpy")
    raise

# Synthetic tabular classification: informative features + redundant noise
RANDOM_STATE = 42
X, y = make_classification(
    n_samples=8_000,
    n_features=24,
    n_informative=12,
    n_redundant=6,
    n_classes=2,
    random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Shapes:", X_train.shape, X_test.shape, "positives in train:", int(y_train.sum()))

### XGBoost

`XGBClassifier` uses depth-wise tree growth by default. We time `fit` with `time.perf_counter()` for a fair wall-clock comparison on this machine.

In [ ]:
try:
    import xgboost as xgb
except ImportError:
    print("Install: pip install xgboost")
    raise

t0 = time.perf_counter()
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)
xgb_clf.fit(X_train, y_train)
xgb_sec = time.perf_counter() - t0
xgb_acc = accuracy_score(y_test, xgb_clf.predict(X_test))
print(f"XGBoost  accuracy={xgb_acc:.4f}  train_time={xgb_sec:.3f}s")

### LightGBM

`LGBMClassifier` often shines on large datasets thanks to histogram-based learning and leaf-wise splits. Parameters are kept in the same ballpark as XGBoost for a rough apples-to-apples demo (not a rigorous benchmark).

In [ ]:
try:
    import lightgbm as lgb
except ImportError:
    print("Install: pip install lightgbm")
    raise

t0 = time.perf_counter()
lgb_clf = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=-1,
    num_leaves=31,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)
lgb_clf.fit(X_train, y_train)
lgb_sec = time.perf_counter() - t0
lgb_acc = accuracy_score(y_test, lgb_clf.predict(X_test))
print(f"LightGBM accuracy={lgb_acc:.4f}  train_time={lgb_sec:.3f}s")

### CatBoost (silent)

`CatBoostClassifier` avoids logging chatter with `verbose=False`. On this purely numeric synthetic set there are no categorical columns—CatBoost still trains fine; its categorical handling matters more on real business tables.

In [ ]:
try:
    from catboost import CatBoostClassifier
except ImportError:
    print("Install: pip install catboost")
    raise

t0 = time.perf_counter()
cb_clf = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.08,
    random_seed=RANDOM_STATE,
    verbose=False,
    thread_count=-1,
)
cb_clf.fit(X_train, y_train)
cb_sec = time.perf_counter() - t0
cb_acc = accuracy_score(y_test, cb_clf.predict(X_test))
print(f"CatBoost accuracy={cb_acc:.4f}  train_time={cb_sec:.3f}s")

### Comparison: accuracy, time, feature importance

We align **feature importance** by index (synthetic columns `f0`…). Each library exposes a different API; we take the first six indices for a compact view.

In [ ]:
try:
    import pandas as pd
except ImportError:
    print("Install: pip install pandas")
    raise

rows = [
    {"model": "XGBoost", "accuracy": xgb_acc, "train_s": xgb_sec},
    {"model": "LightGBM", "accuracy": lgb_acc, "train_s": lgb_sec},
    {"model": "CatBoost", "accuracy": cb_acc, "train_s": cb_sec},
]
summary = pd.DataFrame(rows).set_index("model")
print(summary.to_string())

# Feature importance (first 6 features)
feat_names = [f"f{i}" for i in range(6)]
xgb_imp = xgb_clf.feature_importances_[:6]
lgb_imp = lgb_clf.feature_importances_[:6]
cb_imp = cb_clf.get_feature_importance()[:6]
imp_df = pd.DataFrame(
    {"xgb": xgb_imp, "lgb": lgb_imp, "catboost": cb_imp},
    index=feat_names,
)
print("\nTop-index importances (normalized scales differ by library):")
print(imp_df.to_string())

## Hyperparameter tuning (brief)

Gradient-boosted trees share levers: **number of trees** (`n_estimators` / `iterations`), **learning rate**, **tree depth** (or `num_leaves` in LightGBM), **subsample** and **column subsample** for stochasticity, and **early stopping** on a validation fold to avoid hand-picking iteration counts.

Practical workflow: start with library defaults or a modest grid, use **cross-validation** or a hold-out set, and prefer **early stopping** plus a learning-rate sweep over exhaustive huge grids. Libraries also offer Bayesian / sequential tuning integrations (Optuna, Hyperopt, etc.) when budgets allow.

## Conclusion

- **XGBoost** is a safe default when you need broad docs, tooling, and deployment stories; regularization knobs are explicit.
- **LightGBM** is worth trying when training speed or memory on large numeric tables dominates; validate leaf-wise depth to control overfitting.
- **CatBoost** shines when you have **high-cardinality categoricals** and want strong out-of-the-box behavior with less manual encoding.

No single library wins every dataset—**measure on your data** with the same validation protocol and production constraints (latency, model size, GPU/CPU).